# Standardize Content

Extracts textual content from Bronze files, standardizes the extracted content and file metadata, generates stable content identifiers, preserves source lineage, and writes candidate content documents to the Work layer for validation before promotion to Silver.

## Input Parameters

In [1]:
p_source_container = "DataOps_Control"
p_source_execution_step_id = 116

StatementMeta(, 154ebc18-28cb-4404-99dd-d8c23b59aef9, 3, Finished, Available, Finished, False)

## Imports

In [2]:
from pyspark.sql import functions as F

StatementMeta(, 154ebc18-28cb-4404-99dd-d8c23b59aef9, 4, Finished, Available, Finished, False)

## Configuration

In [3]:
supported_extensions = [
    ".md",
    ".sql",
    ".txt"
]

StatementMeta(, 154ebc18-28cb-4404-99dd-d8c23b59aef9, 5, Finished, Available, Finished, False)

## Load Supported Bronze Files

Loads Bronze file metadata for the selected source execution and keeps only file types supported by the current text-extraction process.

In [ ]:
def load_supported_bronze_files(
    source_container,
    source_execution_step_id,
    supported_extensions
):
    """
    Loads Bronze content metadata for the selected source execution
    and keeps only file types supported by the current text-extraction process.
    """

    return (
        spark.table("bronze.content_files")
        .filter(
            (F.col("source_container") == source_container) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_container",
            "source_path",
            "bronze_path",
            "file_name",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
        .withColumn(
            "file_extension",
            F.lower(
                F.regexp_extract(
                    F.col("file_name"),
                    r"(\.[^.]+)$",
                    1
                )
            )
        )
        .filter(
            F.col("file_extension").isin(supported_extensions)
        )
    )

## Extract File Content

Reads the eligible Bronze files using Spark, extracts their textual content, and joins the physical files back to the registered Bronze metadata.

In [ ]:
def extract_file_content(
    supported_files,
    source_container,
    source_execution_step_id
):
    """
    Reads Bronze files for the selected source execution and joins
    their textual content with the registered Bronze file metadata.
    """

    lakehouse_path = (
        f"Files/bronze/github/"
        f"{source_container}/"
        f"execution_step_id={source_execution_step_id}"
    )

    extracted_content = (
        spark.read
        .format("binaryFile")
        .option("recursiveFileLookup", "true")
        .load(lakehouse_path)
        .select(
            F.concat(
                F.lit("Files/"),
                F.regexp_extract(
                    F.col("path"),
                    r"/Files/(.*)$",
                    1
                )
            ).alias("bronze_path"),
            F.col("content")
            .cast("string")
            .alias("content")
        )
    )

    return (
        supported_files.alias("m")
        .join(
            extracted_content.alias("c"),
            F.col("m.bronze_path") == F.col("c.bronze_path"),
            "inner"
        )
        .select(
            "m.*",
            "c.content"
        )
    )

## Standardize Content

Standardizes file metadata and document content, derives a stable content path, and generates the canonical content identifier.

In [ ]:
def standardize_content(
    extracted_content_documents
):
    """
    Standardizes file metadata and content, derives the stable content path,
    and generates the canonical content identifier.
    """

    return (
        extracted_content_documents

        .withColumn(
            "source_path",
            F.trim(F.col("source_path"))
        )

        .withColumn(
            "bronze_path",
            F.trim(F.col("bronze_path"))
        )

        .withColumn(
            "file_name",
            F.trim(F.col("file_name"))
        )

        .withColumn(
            "file_extension",
            F.lower(
                F.trim(F.col("file_extension"))
            )
        )

        .withColumn(
            "content",
            F.when(
                F.trim(F.col("content")) == "",
                F.lit(None)
            ).otherwise(F.col("content"))
        )

        # Remove the execution-specific path segment so the same source
        # document keeps the same canonical identity across executions.
        .withColumn(
            "content_path",
            F.regexp_replace(
                F.col("bronze_path"),
                r"/execution_step_id=[^/]+",
                ""
            )
        )

        .withColumn(
            "content_id",
            F.sha2(
                F.col("content_path"),
                256
            )
        )
    )

## Prepare Work Content

Selects the standardized fields required by `work.content_documents` and prepares the candidate records for validation.

In [ ]:
def prepare_work_content(
    standardized_content
):
    """
    Selects the standardized content fields required by the Work layer.
    """

    return (
        standardized_content
        .select(
            "content_id",
            "source_container",
            "source_path",
            "bronze_path",
            "content_path",
            "file_name",
            "file_extension",
            "content",
            "source_execution_step_id"
        )
    )

## Write Content to Work

Replaces the current Work content snapshot for the selected source container and returns the number of extracted records for downstream reconciliation.

In [ ]:
def write_work_content(
    work_content_documents,
    source_container
):
    """
    Replaces the current Work content snapshot for the selected
    source container.
    """

    (
        work_content_documents.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_container = '{source_container}'"
        )
        .saveAsTable("work.content_documents")
    )

## Execute Standardization Process

Orchestrates the content-standardization workflow for the selected Bronze source execution.

In [ ]:
supported_files = load_supported_bronze_files(
    source_container=p_source_container,
    source_execution_step_id=p_source_execution_step_id,
    supported_extensions=supported_extensions
)

extracted_content_documents = extract_file_content(
    supported_files=supported_files,
    source_container=p_source_container,
    source_execution_step_id=p_source_execution_step_id
)

standardized_content = standardize_content(
    extracted_content_documents=extracted_content_documents
)

work_content_documents = prepare_work_content(
    standardized_content=standardized_content
)

write_work_content(
    work_content_documents=work_content_documents,
    source_container=p_source_container
)